# Batch Photometry Workflow (No PowerShell Required)

This notebook runs the maintained scripts using the active Jupyter kernel. Start JupyterLab from the `photometry` environment in Anaconda Navigator, then work from top to bottom. Long-running steps are disabled until their `RUN_...` switch is changed to `True`.

## 1. Find the repository

This works when the notebook is opened from either the repository root or the `notebooks` directory.

In [ ]:
from pathlib import Path
import csv
import subprocess
import sys

PROJECT_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the repository root.')

print('Repository:', PROJECT_ROOT)
print('Python:', sys.executable)

## 2. Install the project into this kernel

Set `INSTALL_PROJECT = True` the first time this environment uses the repository. Core analysis does not require the optional NeMoS/JAX dependencies.

In [ ]:
INSTALL_PROJECT = False

if INSTALL_PROJECT:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)],
        check=True,
    )
else:
    print('Installation skipped. Set INSTALL_PROJECT = True to install.')

## 3. Configure data and output paths

Change `DATA_ROOT` if the raw/processed sessions are stored somewhere other than `Z:\Photometry`.

In [ ]:
DATA_ROOT = Path(r'Z:\Photometry')
ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
MANIFEST = ANALYSIS_DIR / 'sessions.csv'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print('Data root:', DATA_ROOT)
print('Manifest:', MANIFEST)
print('Data root is accessible:', DATA_ROOT.exists())

## 4. Create the session manifest

Replace the example rows below. Use six-digit `YYMMDD` dates, integer run numbers, and photoreceiver channel 1 or 2. Set `WRITE_MANIFEST = True` only after reviewing the list. The `analysis` directory is intentionally excluded from Git.

In [ ]:
SESSIONS = [
    {'mouse': 'DK21', 'date': '230704', 'run': 1,
     'group': 'control', 'condition': 'naive', 'channel': 1},
    {'mouse': 'DK21', 'date': '230705', 'run': 2,
     'group': 'control', 'condition': 'trained', 'channel': 1},
]
WRITE_MANIFEST = False

if WRITE_MANIFEST:
    with MANIFEST.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(
            stream,
            fieldnames=('mouse', 'date', 'run', 'group', 'condition', 'channel'),
        )
        writer.writeheader()
        writer.writerows(SESSIONS)
    print(f'Wrote {len(SESSIONS)} sessions to {MANIFEST}')
elif MANIFEST.exists():
    print(MANIFEST.read_text(encoding='utf-8'))
else:
    print('Edit SESSIONS and set WRITE_MANIFEST = True.')

## 5. Script runner

The helper below uses this notebook's Python executable, so every command runs in the selected Jupyter environment.

In [ ]:
def run_script(script_name, *arguments):
    command = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / script_name),
        *[str(argument) for argument in arguments],
    ]
    print('Running:', ' '.join(command))
    return subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## 6. Batch preprocessing

Existing processed files are skipped. Set the switch to `True` to run.

In [ ]:
RUN_PREPROCESS = False

if RUN_PREPROCESS:
    run_script(
        'run_preprocess_batch.py',
        '--manifest', MANIFEST,
        '--data-root', DATA_ROOT,
        '--continue-on-error',
    )
else:
    print('Preprocessing skipped.')

## 7. Cue-aligned photometry PSTHs

Creates individual-mouse, group, and condition-comparison SVG/PNG figures from −5 to +20 seconds around cue onset.

In [ ]:
RUN_CUE_PSTH = False

if RUN_CUE_PSTH:
    run_script(
        'run_psth.py',
        '--manifest', MANIFEST,
        '--data-root', DATA_ROOT,
        '--output-dir', ANALYSIS_DIR / 'cue_psth_20s',
        '--event-key', 'cue_onset',
        '--window', -5, 20,
        '--baseline', -5, 0,
        '--normalization', 'zscore',
    )
else:
    print('Cue PSTH skipped.')

## 8. Cue-aligned licking-rate PSTHs

In [ ]:
RUN_LICKING_PSTH = False

if RUN_LICKING_PSTH:
    run_script(
        'run_psth.py',
        '--manifest', MANIFEST,
        '--data-root', DATA_ROOT,
        '--output-dir', ANALYSIS_DIR / 'licking_psth_20s',
        '--event-key', 'cue_onset',
        '--signal', 'licking',
        '--window', -5, 20,
        '--dt', 0.1,
        '--normalization', 'none',
    )
else:
    print('Licking PSTH skipped.')

## 9. Mouse-level response statistics

Choose the response window and metrics before comparing conditions.

In [ ]:
RUN_STATISTICS = False

if RUN_STATISTICS:
    run_script(
        'run_psth_statistics.py',
        '--manifest', MANIFEST,
        '--data-root', DATA_ROOT,
        '--output-dir', ANALYSIS_DIR / 'statistics' / 'cue',
        '--event-key', 'cue_onset',
        '--baseline', -5, 0,
        '--response-window', 0, 2,
        '--metrics', 'mean', 'auc', 'peak', 'peak_latency',
        '--test', 'auto',
    )
else:
    print('Statistics skipped.')

## 10. Lick-bout-aligned Astrocyte photometry sorted by Ensure delivery

This excludes delivery latencies below zero and writes to a new output directory. Change `--group` for a differently named cohort.

In [ ]:
RUN_LICKBOUT_DELIVERY = False

if RUN_LICKBOUT_DELIVERY:
    run_script(
        'run_lickbout_delivery_analysis.py',
        '--manifest', MANIFEST,
        '--data-root', DATA_ROOT,
        '--output-dir',
        ANALYSIS_DIR / 'astrocyte_lickbout_delivery_nonnegative_20s',
        '--group', 'Astrocyte',
        '--window', -5, 20,
        '--baseline', -5, 0,
        '--minimum-delivery-latency', 0,
        '--normalization', 'zscore',
    )
else:
    print('Lick-bout/delivery analysis skipped.')

## Outputs

All generated tables and figures are placed under `analysis/`, which is excluded from Git. Figures are saved as editable SVG and PNG by default. If a command fails, read its final error message first; the scripts validate missing files, required session keys, and invalid parameters explicitly.